# Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from PIL import Image

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch_directml
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.metrics import confusion_matrix, classification_report

# Part 1 - Data preprocessing
preprocessing the training set

In [2]:
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# Аугментация
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomAffine(degrees=10, shear=0.2, scale=(0.8,1.2)),   # поворот
    transforms.RandomResizedCrop(128, scale=(0.8,1.2)), # zoom_range
    transforms.RandomHorizontalFlip(),               # horizontal_flip
    transforms.ToTensor(),                           # перевод в тензор
    transforms.Normalize((0.5,), (0.5,))             # нормализация
])

# Для теста — только resize + нормализация
test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Датасеты (аналог flow_from_directory)
train_dataset = datasets.ImageFolder('dataset/training_set', transform=train_transform)
test_dataset  = datasets.ImageFolder('dataset/test_set', transform=test_transform)

# DataLoader (аналог batch_size=32)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=4, persistent_workers=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=4, persistent_workers=True)

# Part 2 - Building the CNN

In [3]:

dml = torch_directml.device()

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # Шаг 1 - Свертка
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
        # Шаг 2 - Пулинг
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Второй сверточный слой
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3)
        # Шаг 3 - Flatten → реализуется через torch.flatten в forward
        # Шаг 4 - Полносвязный слой
        self.fc1 = nn.Linear(32 * 30 * 30, 128)
        # Шаг 4.1 - Dropout для регуляризации
        self.dropout = nn.Dropout(0.5)
        # Шаг 5 - Выходной слой
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))   # свертка + активация
        x = self.pool(x)             # пулинг
        x = F.relu(self.conv2(x))   # свертка + активация
        x = self.pool(x)             # пулинг
        x = torch.flatten(x, 1)     # аналог Flatten
        x = F.relu(self.fc1(x))     # полносвязный слой + активация
        x = self.dropout(x)          # dropout — только при train, отключается при eval
        x = self.fc2(x)              # выходной слой (логиты)
        return x

# Part 3 - Trainig the CNN

In [ ]:


history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

model = CNN().to(dml)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, 
                             momentum=0.9, weight_decay=1e-4, nesterov=True)

for epoch in range(25):
    # train
    model.train()
    running_loss, correct = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/25", ncols=80)

    for inputs, labels in pbar:
        inputs, labels = inputs.to(dml), labels.to(dml).long()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc  = correct / len(train_dataset)

    # validation
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(dml), labels.to(dml).long()
            outputs = model(inputs)
            val_loss    += criterion(outputs, labels).item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss = val_loss / len(test_loader)
    val_acc  = val_correct / len(test_dataset)

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    tqdm.write(f"accuracy: {train_acc:.4f} | loss: {train_loss:.4f} | val_accuracy: {val_acc:.4f} | val_loss: {val_loss:.4f}")

Epoch  1/25: 100%|████████████████████████████| 250/250 [00:19<00:00, 13.12it/s]


accuracy: 0.5717 | loss: 0.6786 | val_accuracy: 0.6405 | val_loss: 0.6356


Epoch  2/25: 100%|████████████████████████████| 250/250 [00:05<00:00, 41.71it/s]


accuracy: 0.6268 | loss: 0.6357 | val_accuracy: 0.6765 | val_loss: 0.6126


Epoch  3/25: 100%|████████████████████████████| 250/250 [00:05<00:00, 45.39it/s]


...

Epoch 16/25: 100%|████████████████████████████| 250/250 [00:05<00:00, 45.45it/s]


accuracy: 0.7936 | loss: 0.4470 | val_accuracy: 0.7885 | val_loss: 0.4912


Epoch 17/25: 100%|████████████████████████████| 250/250 [00:05<00:00, 45.54it/s]


accuracy: 0.7959 | loss: 0.4385 | val_accuracy: 0.8120 | val_loss: 0.4229


Epoch 18/25:  50%|██████████████              | 126/250 [00:02<00:02, 45.46it/s]

# Part 4 - Metrics

### Confusion Matrix

In [ ]:

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs.to(dml))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

cm         = confusion_matrix(all_labels, all_preds)
cm_norm    = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
class_names = train_dataset.classes

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.suptitle('Confusion matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
best_val_acc  = max(history['val_acc'])
best_epoch    = history['val_acc'].index(best_val_acc) + 1
final_val_acc = history['val_acc'][-1]
final_train   = history['train_acc'][-1]
gap           = final_train - final_val_acc

print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ:\n')
print(f'Финальная train accuracy:  {final_train:.4f} ({final_train*100:.2f}%)')
print(f'Финальная val accuracy:    {final_val_acc:.4f} ({final_val_acc*100:.2f}%)')
print(f'Лучшая val accuracy:       {best_val_acc:.4f} (эпоха {best_epoch})')
print(f'Разрыв train/val (gap):    {gap:.4f}')

# Part 5 - prediction test

In [ ]:

img_path = 'dataset/single_prediction/cat_or_dog_2.jpg'
image = Image.open(img_path).convert('RGB')

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_image = transform(image).unsqueeze(0).to(dml)

model.eval()
with torch.no_grad():
    output = model(test_image)

probs = torch.softmax(output, dim=1)
prob_dog = probs[0][1].item()
prob_cat = probs[0][0].item()

if prob_dog > prob_cat:
    prediction = 'dog'
    probability = prob_dog
else:
    prediction = 'cat'
    probability = prob_cat

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image)
ax.set_title(f"Prediction: {prediction}\nProbability: {probability*100:.2f} %", fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), 'cat_dog_cnn_final_model.pth')
print('\nМодель сохранена: cat_dog_cnn_final_model.pth')